# Spaceship Titanic — Controlled Feature Ablation

**Purpose:** Compare core original predictors against the same predictors plus an engineered feature block.

**Design:** Identical stratified 80/20 split (`random_state=42`), fixed Random Forest settings, and preprocessing fitted within each training partition. Passenger group and surname counts use combined train+test metadata without target labels; this is transductive feature construction and requires group-aware robustness testing.

## 1. Upload and load data

Upload `train.csv` and `test.csv` to the Colab working directory using the Files panel, or place them in `data/`.

In [3]:
from pathlib import Path
import pandas as pd

def locate_csv(filename):
    for p in [Path(filename), Path("data") / filename]:
        if p.exists():
            return p
    raise FileNotFoundError(f"Upload {filename} to the working directory or put it in data/.")

train = pd.read_csv(locate_csv("train.csv"))
test = pd.read_csv(locate_csv("test.csv"))
print("Training:", train.shape, "| Test:", test.shape)
display(train.head())


Training: (8693, 14) | Test: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 2. Feature engineering and evaluation functions

The imputation and one-hot encoding steps are inside the sklearn pipeline, so they are fitted on the training partition only.

In [4]:
"""Reproduce the controlled Spaceship Titanic feature-ablation experiment.

Run from the repository root after installing requirements.txt:
    python src/feature_ablation.py --train data/train.csv --test data/test.csv

This experiment uses the same stratified 80/20 holdout indices and the same
Random Forest settings for both feature sets. Passenger group/surname counts
are target-free aggregates from train+test passenger metadata; this is a
transductive feature construction and should be evaluated with group-aware
validation before final model selection.
"""
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def build_engineered_features(frame, group_counts, surname_counts):
    x = frame.copy()
    x["TotalSpend"] = x[SPEND].sum(axis=1, min_count=1)
    x["LogTotalSpend"] = np.log1p(x["TotalSpend"])
    x["SpendObservedCount"] = x[SPEND].notna().sum(axis=1)
    x["AnySpend"] = np.where(x["TotalSpend"].notna(), (x["TotalSpend"] > 0).astype(float), np.nan)
    x["NoSpend"] = np.where(x["TotalSpend"].notna(), (x["TotalSpend"] == 0).astype(float), np.nan)
    parts = x["PassengerId"].str.split("_", expand=True)
    x["PassengerNumber"] = pd.to_numeric(parts[1], errors="coerce")
    x["GroupSize"] = parts[0].map(group_counts)
    cabin = x["Cabin"].fillna("").str.split("/", expand=True)
    x["CabinDeck"] = cabin[0].replace("", np.nan)
    x["CabinNumber"] = pd.to_numeric(cabin[1], errors="coerce")
    x["CabinSide"] = cabin[2].replace("", np.nan)
    x["AgeGroup"] = pd.cut(
        x["Age"], [-np.inf, 12, 18, 30, 50, np.inf],
        labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
    ).astype("object")
    last_name = x["Name"].fillna("").str.split().str[-1].replace("", np.nan)
    x["SurnameCount"] = last_name.map(surname_counts)
    return x


def evaluate(x_train, x_valid, y_train, y_valid):
    categorical = x_train.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric = x_train.select_dtypes(exclude=["object", "category"]).columns.tolist()
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median", add_indicator=True), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical)
    ])
    model = Pipeline([
        ("preprocess", prep),
        ("random_forest", RandomForestClassifier(
            n_estimators=250, min_samples_leaf=2, max_features="sqrt",
            n_jobs=-1, random_state=42
        ))
    ])
    model.fit(x_train, y_train)
    pred = model.predict(x_valid)
    return {
        "accuracy": float(accuracy_score(y_valid, pred)),
        "precision": float(precision_score(y_valid, pred)),
        "recall": float(recall_score(y_valid, pred)),
        "f1": float(f1_score(y_valid, pred)),
        "confusion_matrix": confusion_matrix(y_valid, pred).tolist()
    }


# Function definitions above are shared with src/feature_ablation.py.


## 3. Create comparable feature sets and one shared split

In [5]:
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
y = train["Transported"].astype(int)
combined = pd.concat([train.drop(columns="Transported"), test], ignore_index=True)
group_counts = combined["PassengerId"].str.split("_").str[0].value_counts()
surname = combined["Name"].fillna("").str.split().str[-1].replace("", pd.NA)
surname_counts = surname.value_counts()
engineered = build_engineered_features(train, group_counts, surname_counts)

core_columns = ["HomePlanet", "CryoSleep", "Destination", "Age", "VIP"] + SPEND
engineered_columns = core_columns + [
    "TotalSpend", "LogTotalSpend", "SpendObservedCount", "AnySpend", "NoSpend",
    "PassengerNumber", "GroupSize", "CabinDeck", "CabinNumber", "CabinSide",
    "AgeGroup", "SurnameCount"
]
x_core = train[core_columns].copy()
x_engineered = engineered[engineered_columns].copy()

from sklearn.model_selection import train_test_split
train_idx, valid_idx = train_test_split(
    train.index, test_size=0.2, random_state=42, stratify=y
)
y_train, y_valid = y.loc[train_idx], y.loc[valid_idx]
print(f"Training rows: {len(train_idx)} | Validation rows: {len(valid_idx)}")
print("Both feature sets use exactly the same row indices.")


Training rows: 6954 | Validation rows: 1739
Both feature sets use exactly the same row indices.


## 4. Run the controlled ablation

Both runs use Random Forest: 250 trees, `min_samples_leaf=2`, `max_features='sqrt'`, `random_state=42`.

In [6]:
core_metrics = evaluate(x_core.loc[train_idx], x_core.loc[valid_idx], y_train, y_valid)
eng_metrics = evaluate(x_engineered.loc[train_idx], x_engineered.loc[valid_idx], y_train, y_valid)

results = pd.DataFrame([
    {"Feature_Set": "Core original predictors", **{k: core_metrics[k] for k in ["accuracy", "precision", "recall", "f1"]}},
    {"Feature_Set": "Core + engineered predictors", **{k: eng_metrics[k] for k in ["accuracy", "precision", "recall", "f1"]}},
])
display(results.round(4))
print(f"Accuracy change: {(eng_metrics['accuracy']-core_metrics['accuracy'])*100:.2f} percentage points")
print("Core confusion matrix:", core_metrics["confusion_matrix"])
print("Engineered confusion matrix:", eng_metrics["confusion_matrix"])


,Feature_Set,accuracy,precision,recall,f1
0,Core original predictors,0.7872,0.7875,0.7911,0.7893
1,Core + engineered predictors,0.8056,0.8172,0.7911,0.8039


Accuracy change: 1.84 percentage points
Core confusion matrix: [[676, 187], [183, 693]]
Engineered confusion matrix: [[708, 155], [183, 693]]


## 5. Save results

In [7]:
from pathlib import Path
import json
outdir = Path("reports")
outdir.mkdir(exist_ok=True)
results.to_csv(outdir / "feature_ablation.csv", index=False)
details = {
    "Core original predictors": core_metrics,
    "Core + engineered predictors": eng_metrics,
    "design": {
        "split": "stratified 80/20", "random_state": 42,
        "model": "RandomForestClassifier(n_estimators=250, min_samples_leaf=2, max_features='sqrt', random_state=42)",
        "group_surname_counts": "Computed from train+test unlabeled metadata; transductive."
    }
}
(outdir / "feature_ablation_details.json").write_text(json.dumps(details, indent=2))
print("Saved reports/feature_ablation.csv and reports/feature_ablation_details.json")


Saved reports/feature_ablation.csv and reports/feature_ablation_details.json


## 6. Interpretation and limitations

This tests the combined engineered feature block on one holdout split; it does not identify which individual feature helps. The result is local validation, not a Kaggle leaderboard score.

**Next:** run group-aware validation, test feature groups separately (cabin, passenger group, spending, age/surname), and report fold mean and variability before final model selection.